# Nemotron: tune context, request load and output length

**You are here → 02 — Nemotron optimization.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| **→ 2. Nemotron optimization (current)** | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| Optional CPU portability | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| Optional compatibility study | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

Chapter 01 tested caching and weight precision with Qwen. Here we'll keep the
clinical SQL application and use **Nemotron 3 Nano 4B through vLLM**. Nemotron
combines attention layers with Mamba layers, which carry recurrent state as tokens
are processed. That affects which cache and precision settings the runtime supports.

We'll start from FP16 weights with FP32 Mamba state, then vary **schema context,
prefix caching where supported, concurrent requests, scheduler capacity, reasoning
and output limits**. For each change, we'll ask which work it reduces and whether
the application still completes a useful answer. Read
[chapter 01](01_qwen_cache_quantization.ipynb) for the shared timing and caching concepts.

You can work through this chapter in two ways. **Read the completed experiment:**
choose **Run All** in the [notebook environment](../SETUP.md) to display saved
measurements. **Run a new experiment:** use the [Live setup](../tutorial/RUN_ALL.md)
with a compatible GPU. The default reads results only; it downloads no weights
and starts no inference server.

The GPU measurements in this chapter used an **NVIDIA Quadro RTX 5000 with
16 GiB VRAM**. We use synthetic clinical-trial questions to connect speed with
SQL outcomes. These experiments help us understand this workload, rather than
predict performance for every application.


## 1. Choose how to run the experiment

The next cell loads the configuration. Check the printed mode so you know
whether later cells are reading a completed run or collecting new results.

| Mode | What the cells do |
|---|---|
| **Replay (default)** | Read the supplied results and rebuild their tables; no inference runs |
| Plan | Show settings and commands for a new experiment; no inference runs |
| Live | Start the configured server, make requests and save new measurements |

For Plan or Live, create `tutorial/config/vllm-nemotron.local.json` using
[the run guide](../tutorial/RUN_ALL.md). Put your paths and settings there. Editing
that config affects a new run; it doesn't change the measurements read in Replay.

We use `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16` through native vLLM, with weights loaded as **FP16**.
The source checkpoint's BF16 name is not the runtime precision: weights load as FP16, while Mamba SSM state stays FP32.

There are **four distinct questions**, repeated five times for serving measurements.
The separate application evaluation includes planning, SQL checks, execution, and up
to one repair. Twenty repeated requests are not twenty independent quality examples.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.workflow import Study
from clinical_trial_mvp.tutorial.teaching import server_example, request_example, response_example, interpret
if 'study' in globals():
    study.close()
study = Study('vllm', 'nemotron', root=ROOT)
results = study.results
print('Mode:', study.mode)
print('Model:', 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16')
print(study.prepare())


Mode: replay
Model: nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16
Replay: recorded Dell measurements; no model download, server, GPU or network calls.


## 2. Decide which part of inference we're changing

We'll group the experiments into input work, serving capacity and output work.
This helps us explain a speed change instead of treating every flag as the
same kind of optimization. Think about the expected timing and quality effects
before reading each result.


We begin with FP16 weights and fixed FP32 Mamba state. Then we test prefix
caching, retrieved schema, concurrency, scheduler capacity, reasoning, and output caps.
Each comparison changes a selected factor; we did not measure a combined “best of everything” configuration.

Hybrid prefix caching needs support from the installed backend. Enable
`prefix_supported=true` only after verifying that support. AWQ is a separate,
optional extension: we did not obtain a validated Nemotron AWQ checkpoint.
Unsupported rows should be skipped or marked unavailable—not scored as zero throughput.

With reasoning enabled, intermediate thought consumes output tokens. TTFT may
then mark the first thought token, not the first user-facing answer.


## 3. Check that the model can complete a SQL request

Before comparing speed, check that the hybrid model and runtime can work
with the application. This **smoke test** sends a few questions through planning,
SQL generation and execution. It can catch an integration failure before we
spend time on a larger run; it doesn't replace the later quality comparison.

In Replay, the next cell shows the saved smoke result when one is available.
Plan makes no model calls. Live runs the check and stops dependent experiments
if the required check fails. Read the status and any error before continuing.


In [2]:
smoke = study.smoke()
if isinstance(smoke, dict):
    print(smoke.get('_recorded', {}).get('label', study.mode), smoke.get('status'), smoke.get('reason', ''))
else:
    print(smoke)


replay unavailable No such measurement was recorded in the Dell archive; no values estimated.


## 4. Run the comparisons

We'll establish a baseline, then change a selected setting in each comparison.
Before running a cell, note what changes and what stays fixed. Afterward, compare
request timing with the generated SQL and its result check. An experiment finishing
successfully doesn't mean every answer it produced was correct.


### A. Input work: how much do we process again?

First we'll keep the request load fixed and change how the model handles input:
reuse a prefix, or provide less schema. Those are different interventions—one
reuses information, while the other removes some of it.


#### Start with the FP16 baseline

Run this cell to establish the full-schema reference. We'll use its timing and
quality to interpret the following changes, keeping FP32 Mamba state fixed.


In [3]:
result = study.run('fp16_baseline')
print(result.get('_recorded', {}).get('label', study.mode), 'fp16_baseline:', result['status'])
display(Markdown(interpret(study, 'fp16_baseline')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'fp16_baseline')))


recorded Dell measurement fp16_baseline: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T002501_223219Z.json](../experiment-tracker/mvp4/nemotron/20261107T002501_223219Z.json)

**Prediction:** Establish executable SQL and separate application quality from request speed.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 4.251/4.892 s; aggregate output: 36.622 tok/s.

**Explanation:** Transport timing and application correctness measure different stages.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use this baseline to judge useful completed tasks, not latency alone.

### Connect the server settings to the request

Start with the server: which checkpoint is loaded, what precision does it use,
and what context and scheduling limits apply? The next cell displays those settings
and a launch command so you can connect the experiment config to the serving process.
In Plan, it shows the proposed settings for your new run.

The [vLLM launcher](../src/clinical_trial_mvp/serving/launch.py) contains the implementation.


In [4]:
display(JSON(server_example(study, 'fp16_baseline')))


<IPython.core.display.JSON object>

**Next, inspect the request.** The model receives a prompt, generation limits
and, where enabled, a JSON response schema. Look for the shared database context,
the question-specific text and the requested output format in the next cell.
This lets us see what the application asks the model to do before judging its answer.

This inspection displays the example without sending another request.
The [request builder](../src/clinical_trial_mvp/serving/client.py) defines the payload.


In [5]:
display(JSON(request_example(study, 'fp16_baseline')))


<IPython.core.display.JSON object>

**Finally, follow the answer.** The next cell shows the generated plan and SQL,
any repair attempt, and the comparison with the reference query. Check the
question's filters, grouping and counted population as well as the score.
Matching rows on this database is useful, but it doesn't prove that two queries
express the same logic for every possible dataset.

You can inspect the [plan and SQL checks](../src/clinical_trial_mvp/agent/analyst.py)
and [reference-result scorer](../src/clinical_trial_mvp/evaluation/scorers.py).


In [6]:
display(JSON(response_example(study, 'fp16_baseline')))


<IPython.core.display.JSON object>

#### Reuse a prefix with the hybrid backend

Can Nemotron reuse the shared schema prefix on this backend? Compare cold and
warm requests in the cache report. If the backend is unsupported, the row should
say so; the remaining experiments can still proceed.


In [7]:
result = study.run('prefix_cache')
print(result.get('_recorded', {}).get('label', study.mode), 'prefix_cache:', result['status'])
display(Markdown(interpret(study, 'prefix_cache')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'prefix_cache')))


recorded Dell measurement prefix_cache: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T023148_583457Z.json](../experiment-tracker/mvp4/nemotron/20261107T023148_583457Z.json)

**Prediction:** Shared-prefix reuse should reduce warm prefill latency more than total generation time.

**Observation:** Cold → warm TTFT p50: 1.101 → 0.300 s. Request E2E p50: 4.327 → 3.529 s. Direct SQL result match: 15/20 cold; 15/20 warm.

**Explanation:** The paired phase records are consistent with avoiding repeated prefill; generation still consumes time.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Consider caching for repeated schema/system prefixes after checking both phase quality counts.

#### Send only the retrieved schema

Now reduce the schema text instead of reusing it. Watch TTFT, but also inspect
the SQL: did the smaller prompt omit a required join, definition or denominator?
A latency reduction is not automatically an application improvement.


In [8]:
result = study.run('schema_retrieved')
print(result.get('_recorded', {}).get('label', study.mode), 'schema_retrieved:', result['status'])
display(Markdown(interpret(study, 'schema_retrieved')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'schema_retrieved')))


recorded Dell measurement schema_retrieved: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T024417_765664Z.json](../experiment-tracker/mvp4/nemotron/20261107T024417_765664Z.json)

**Prediction:** Less context may reduce input work but can remove definitions required for correct SQL.

**Observation:** Application result match: 1/4; SQL executed: 1/4. Request E2E p50/p95: 2.593/3.546 s; aggregate output: 46.300 tok/s. Baseline application result match: 2/4.

**Explanation:** Context reduction changes the information available to the model, not just compute cost.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Retain needed clinical definitions; accept reduced context only with adequate task quality.

### B. Serving capacity: overlapping requests versus scheduler limits

Concurrency is how many requests clients send at once. Scheduler capacity limits
how many sequences the server can work on together. We'll vary these separately;
neither setting tells us the actual GPU batch occupancy.


#### Send two requests at a time

Increase client concurrency to two while retaining scheduler capacity four.
Compare per-request latency and aggregate output throughput with concurrency one.
More completed tokens per second can coexist with slower individual requests.


In [9]:
result = study.run('concurrency_2')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_2:', result['status'])
display(Markdown(interpret(study, 'concurrency_2')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_2')))


recorded Dell measurement concurrency_2: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T025050_224975Z.json](../experiment-tracker/mvp4/nemotron/20261107T025050_224975Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 5.362/7.011 s; aggregate output: 56.243 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

#### Send four requests at a time

Increase concurrency to four with the same scheduler capacity. Does throughput
continue to improve, and what happens to the wait for an individual answer?
The useful operating point depends on both, not just peak tokens per second.


In [10]:
result = study.run('concurrency_4')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_4:', result['status'])
display(Markdown(interpret(study, 'concurrency_4')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_4')))


recorded Dell measurement concurrency_4: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T030851_257660Z.json](../experiment-tracker/mvp4/nemotron/20261107T030851_257660Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 7.699/10.557 s; aggregate output: 76.876 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

#### Keep four clients, but allow one sequence at a time

Now hold client concurrency at four and reduce `max-num-seqs` to one.
Compare with the preceding four-client run to isolate scheduler capacity.
Expect queueing to matter; this is not another concurrency-one experiment.


In [11]:
result = study.run('batch_capacity_1')
print(result.get('_recorded', {}).get('label', study.mode), 'batch_capacity_1:', result['status'])
display(Markdown(interpret(study, 'batch_capacity_1')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'batch_capacity_1')))


recorded Dell measurement batch_capacity_1: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T031845_031809Z.json](../experiment-tracker/mvp4/nemotron/20261107T031845_031809Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 16.704/16.748 s; aggregate output: 36.382 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

### C. Output work: how much should the model generate?

Next we'll change the generation budget while returning to the relevant baseline
settings. Extra thinking may help a difficult task, but it also consumes tokens
and time. A larger cap merely permits more output; it doesn't force the model to use it.


#### Let the model reason before answering

Enable thinking and compare both correctness and completion time with the baseline.
Inspect whether the final SQL arrives within the output budget. Earlier thought
tokens can make TTFT look responsive even when the final answer takes much longer.


In [12]:
result = study.run('reasoning_on')
print(result.get('_recorded', {}).get('label', study.mode), 'reasoning_on:', result['status'])
display(Markdown(interpret(study, 'reasoning_on')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'reasoning_on')))


recorded Dell measurement reasoning_on: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T033328_571911Z.json](../experiment-tracker/mvp4/nemotron/20261107T033328_571911Z.json)

**Prediction:** Reasoning/output budgets change generation work and may affect task completion.

**Observation:** Application result match: 2/4; SQL executed: 2/4. Request E2E p50/p95: 9.104/16.138 s; aggregate output: 43.133 tok/s.

**Explanation:** Thoughts consume output tokens; TTFT includes thoughts, and truncated answers remain failures.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use the smallest budget that preserves required task quality; do not optimize output tok/s alone.

#### Limit output to 256 tokens

Try a tighter cap. Do requests finish sooner, or do they end before the plan and
SQL are complete? A truncated answer remains a failure, even if it is fast.


In [13]:
result = study.run('output_256')
print(result.get('_recorded', {}).get('label', study.mode), 'output_256:', result['status'])
display(Markdown(interpret(study, 'output_256')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'output_256')))


recorded Dell measurement output_256: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T034418_092327Z.json](../experiment-tracker/mvp4/nemotron/20261107T034418_092327Z.json)

**Prediction:** Reasoning/output budgets change generation work and may affect task completion.

**Observation:** Application result match: 2/4; SQL executed: 2/4. Request E2E p50/p95: 4.265/4.918 s; aggregate output: 36.422 tok/s.

**Explanation:** Thoughts consume output tokens; TTFT includes thoughts, and truncated answers remain failures.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use the smallest budget that preserves required task quality; do not optimize output tok/s alone.

#### Allow up to 2,048 tokens

Now give the model more room. Check actual output tokens and task results rather
than assuming a larger maximum must increase latency or correctness.


In [14]:
result = study.run('output_2048')
print(result.get('_recorded', {}).get('label', study.mode), 'output_2048:', result['status'])
display(Markdown(interpret(study, 'output_2048')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'output_2048')))


recorded Dell measurement output_2048: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T035411_952257Z.json](../experiment-tracker/mvp4/nemotron/20261107T035411_952257Z.json)

**Prediction:** Reasoning/output budgets change generation work and may affect task completion.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 4.268/4.914 s; aggregate output: 36.491 tok/s.

**Explanation:** Thoughts consume output tokens; TTFT includes thoughts, and truncated answers remain failures.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use the smallest budget that preserves required task quality; do not optimize output tok/s alone.

### Optional: a different weight representation

This last row is not needed for the core Nemotron study. It is available only
when a compatible AWQ derivative has been independently validated.


#### Attempt AWQ only with a verified checkpoint

We did not obtain a compatible Nemotron AWQ artifact in the recorded study.
Expect this extension to remain unavailable unless you supply one with verified
lineage and runtime support. Do not substitute GGUF, FP8 or NVFP4 and call it the same experiment.


In [15]:
result = study.run('awq_stretch')
print(result.get('_recorded', {}).get('label', study.mode), 'awq_stretch:', result['status'])
display(Markdown(interpret(study, 'awq_stretch')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'awq_stretch')))


replay awq_stretch: unavailable


**Prediction:** Four-bit weights reduce weight storage; latency and SQL quality may change in either direction.

**Observation:** No such measurement was recorded in the Dell archive; no values estimated.

**Explanation:** Quantization changes weight representation and kernel behavior; caching addresses prefix reuse.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Benchmark compatible AWQ when weight capacity matters; no native VRAM reduction is established here.

## 5. Which changes helped the application?

Read the summary by experiment group rather than choosing the fastest row overall.
For context changes, check result quality. For concurrency, read throughput beside
per-request latency. For reasoning and output caps, check whether a complete,
useful answer was produced.

TTFT includes the first thought token when reasoning is on. Request E2E and full
application E2E measure different scopes. Human SQL review is pending, and archived
native VRAM readings are unavailable; these records do not establish memory savings.


In [16]:
display(Markdown(study.report()))
display(Markdown(study.cache_report()))


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP4/Nemotron/fp16_baseline · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.05 | 4.25/4.89 | 36.62 | — |
| MVP4/Nemotron/prefix_cache · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 0.32 | 3.58/4.21 | 43.79 | — |
| MVP4/Nemotron/schema_retrieved · recorded Dell measurement | 1/4 | 1/4 | Unreviewed; 4 total | — | 0.20 | 2.59/3.55 | 46.30 | — |
| MVP4/Nemotron/concurrency_2 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.10 | 5.36/7.01 | 56.24 | — |
| MVP4/Nemotron/concurrency_4 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.11 | 7.70/10.56 | 76.88 | — |
| MVP4/Nemotron/batch_capacity_1 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 13.39 | 16.70/16.75 | 36.38 | — |
| MVP4/Nemotron/reasoning_on · recorded Dell measurement | 2/4 | 2/4 | Unreviewed; 4 total | — | 1.06 | 9.10/16.14 | 43.13 | — |
| MVP4/Nemotron/output_256 · recorded Dell measurement | 2/4 | 2/4 | Unreviewed; 4 total | — | 1.06 | 4.27/4.92 | 36.42 | — |
| MVP4/Nemotron/output_2048 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.05 | 4.27/4.91 | 36.49 | — |
| nemotron/awq_stretch · unavailable | — | — | — | — | — | — | — | — |

| Experiment | Application E2E p50/p95 s | Request TTFT p95 s |
|---|---|---|
| fp16_baseline · recorded Dell measurement | 10.711/17.867 | 1.064 |
| prefix_cache · recorded Dell measurement | 8.828/15.832 | 0.327 |
| schema_retrieved · recorded Dell measurement | 8.117/9.086 | 0.267 |
| concurrency_2 · recorded Dell measurement | 10.199/17.928 | 1.677 |
| concurrency_4 · recorded Dell measurement | 10.220/17.924 | 3.390 |
| batch_capacity_1 · recorded Dell measurement | 10.255/17.978 | 14.526 |
| reasoning_on · recorded Dell measurement | 24.388/37.500 | 1.072 |
| output_256 · recorded Dell measurement | 8.861/11.656 | 1.067 |
| output_2048 · recorded Dell measurement | 10.187/17.951 | 1.064 |
| awq_stretch | —/— | — |

| Variant | Phase | SQL match | Semantic | TTFT p50/p95 s | E2E p50/p95 s | TPOT p50 s | tok/s | Peak VRAM GiB |
|---|---|---|---|---|---|---|---|---|
| prefix_cache · recorded Dell measurement | cold | 15/20 | unreviewed | 1.101/1.109 | 4.327/4.968 | 0.021 | 35.945 | — |
| prefix_cache · recorded Dell measurement | warm | 15/20 | unreviewed | 0.300/0.305 | 3.529/4.179 | 0.021 | 44.274 | — |

### Visualize the concurrency trade-off

![Quadro RTX 5000 concurrency results](../tutorial/figures/02_concurrency.png)

This figure shows the saved Quadro RTX 5000 runs, not a new Live run. Use it to relate
per-request delay to throughput, then compare with your own result tables if you
rerun the experiment. [Figure data](../tutorial/figures/figure_data.json) and the
[rendering script](../scripts/tutorial/render_findings.py) are included.


## 6. Close the study

Save the notebook to retain the displayed outputs. Replay leaves the underlying
measurements unchanged; Live saves new uniquely named records rather than
overwriting the archive.

Native servers stop after each experiment row. The next cleanup cell is safe
to repeat. The workflow cleans up its own servers on handled errors and interruptions,
but cannot do so after a kernel or OS crash. In that case, use the recorded
process/container identifier to stop only the server this study started.


In [17]:
print(study.close())


Study servers stopped. Saved records and notebook outputs are retained.


### Where to go next

Continue to [Qwen through NIM](03_qwen_nim.ipynb), or compare deployment choices in
[the results guide](../tutorial/RESULTS.md#what-should-i-choose).

For setup problems, see the [worked troubleshooting examples](../tutorial/CASE_STUDIES.md).
The [original executed notebook](history/dell-2026-11-07/local%20AI%20optimization%20-%20Nemotron.ipynb) retains the original run outputs.
The [SQL review packet](../tutorial/SQL_REVIEW.md) and
[GPU workflow checklist](../tutorial/GPU_VALIDATION.md) describe the remaining validation work.
